In [1]:
%load_ext autoreload
%autoreload 2
import sys, os
nb_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(nb_dir, '..'))
if project_root not in sys.path: sys.path.insert(0, project_root)

# Vllm flow quickly

In [2]:
from slurm_ops.core import (
    start_or_connect,
    job_stat,
    update_ssh_node_config,
    find_free_port,
    get_port_forwarding_command,
)

In [3]:
# ! cp ../ssh_config_templates/* ~/.ssh/
# ! for f in config klone-node-config tillicum-node-config; do sed -i '' 's/deanlcs/<YOUR_CS_ID>/g' ~/.ssh/"$f"; done


In [4]:
! ssh klone-login echo "Connected successfully"

Connected successfully
/mmfs1/home/bbioren/.bashrc: line 19: direnv: command not found


In [12]:
job_name = "remote_dev"
slurm_host = "klone-login"
# Resource presets - pick one based on your task
PRESETS = {
    'quick':     '--account=stf-ckpt --partition=ckpt --qos=ckpt-gpu --gpus=1 --cpus-per-task=4  --mem=20G  --time=00:30:00',
    'afternoon': '--account=stf-ckpt --partition=ckpt --qos=ckpt-gpu --gpus=1 --cpus-per-task=8  --mem=64G  --time=04:00:00',
    'overnight': '--account=stf-ckpt --partition=ckpt --qos=ckpt-gpu --gpus=1 --cpus-per-task=8  --mem=80G  --time=12:00:00',
    'multi_gpu': '--account=stf-ckpt --partition=ckpt --qos=ckpt-gpu --gpus=4 --cpus-per-task=16 --mem=200G --time=1-00:00:00',
}
preset = 'quick'
start_or_connect(job_name, slurm_host, slurm_args=PRESETS[preset])

No running job 'remote_dev' found. Starting salloc...
Run this in your terminal:
ssh -t klone-login "tmux new-session -A -s remote_dev 'salloc --job-name=remote_dev --account=stf-ckpt --partition=ckpt --qos=ckpt-gpu --gpus=1 --cpus-per-task=4  --mem=20G  --time=00:30:00'"


In [11]:
node, job_id = job_stat(job_name, slurm_host)

No running job 'remote_dev' found on klone-login.


TypeError: cannot unpack non-iterable NoneType object

In [7]:
update_ssh_node_config(job_name,host=slurm_host)

Updated /Users/benbioren/.ssh/klone-node-config: Hostname → z3005


'z3005'

In [8]:
! cat /Users/benbioren/.ssh/klone-node-config

Host klone-node
  User bbioren
  Hostname z3005               
  ProxyJump klone-login

In [9]:
print("uv run bash run_vllm_script.sh")

uv run bash run_vllm_script.sh


In [10]:
import os
from openai import OpenAI
from dotenv import load_dotenv

In [ ]:
load_dotenv('../.envrc') # make sure you have a similar .envrc on your local machine
port = os.environ["PORT"]

KeyError: 'PORT'

In [ ]:
_=get_port_forwarding_command(local_port=port, remote_port=port, node=node, host=slurm_host)

ssh -N -f -L 8555:z3001.hyak.local:8555 klone-login


In [ ]:

api_key = os.environ['API_KEY']
model = os.environ['MODEL']

openai_api_base = f"http://localhost:{local_port}/v1"
client = OpenAI(
    api_key=api_key,
    base_url=openai_api_base,
)
completion = client.completions.create(
    model=model,
    prompt="who are you?",
)
print("Completion result:", completion)